# Account Intelligence Pipeline - Analytics
## Win-Rate Analysis & Opportunity Scoring
Open-source intelligence engine using SQLite + Python

In [ ]:
import sys, json, os, math
from pathlib import Path
from collections import Counter, defaultdict

sys.path.insert(0, str(Path('..').resolve()))
from modules.database import *
from modules.analytics import Analytics
from modules.rag import IntelRAG

init_db()
analytics = Analytics(str(Path.cwd() / '..' / 'data' / 'intel.db'))
print('✓ Database connected')

In [ ]:
# Load accounts and score
accounts = query_all_accounts()
print(f'Accounts in pipeline: {len(accounts)}\n')

conn = get_conn()
cur = conn.cursor()
cur.execute("SELECT company, research_json FROM accounts")
full_data = []
for row in cur.fetchall():
    try:
        data = json.loads(row[1])
        full_data.append(data)
    except:
        pass

pipeline = analytics.generate_report(full_data)
for item in pipeline:
    bar = '█' * (item['opportunity_score'] // 5) + '░' * ((100 - item['opportunity_score']) // 5)
    print(f"{item['company']:35s} | {item['opportunity_score']:3d}/100 | {bar} | {item['priority']}")

In [ ]:
# RAG search demo
rag = IntelRAG()
rag.load_from_database(_ih)  # won't work - let's use db_module

import importlib
import modules.database as db_mod
importlib.reload(db_mod)
rag = IntelRAG()
rag.load_from_database(db_mod)

queries = [
    'SAP S/4HANA migration opportunity',
    'supply chain pain points manufacturing',
    'regulatory compliance life sciences',
    'digital transformation retail',
    'NBFC gold loan fintech'
]

print('RAG Search Results:')
for q in queries:
    results = rag.search(q, top_k=2)
    if results:
        print(f'\nQuery: {q}')
        for r in results:
            print(f'  [{r["similarity"]}] {r["source"]}')
            print(f'  {r["snippet"][:100]}...')

In [ ]:
# Pain point heat map by category
cur.execute("""
    SELECT a.company, pp.pain, pp.severity
    FROM pain_points pp
    JOIN accounts a ON a.id = pp.account_id
    ORDER BY a.company
""")
pains = cur.fetchall()

from collections import defaultdict
by_company = defaultdict(list)
for c, p, s in pains:
    by_company[c].append((p, s))

for company, pain_list in sorted(by_company.items()):
    print(f'\n{company} ({len(pain_list)} pain points):')
    for p, s in pain_list:
        severity = '🔴' if s >= 4 else '🟡' if s >= 2 else '🟢'
        print(f'  {severity} {p[:80]}')

In [ ]:
# Transformation signals overview
cur.execute("""
    SELECT a.company, ts.signal_text, ts.opportunity_score
    FROM transformation_signals ts
    JOIN accounts a ON a.id = ts.account_id
    ORDER BY ts.opportunity_score DESC
""")
signals = cur.fetchall()
print('Top Digital Transformation Signals:')
for c, s, score in signals[:10]:
    print(f'  [{score}/10] {c}: {s}')